# Capacity with Facility Data Only

Estimate reported bed counts from Virtue Foundation facility features. This is
the baseline for the two population density notebooks. It uses the same trainer,
facility-group split, and held-out rows. It reads only the facility table.

Open this notebook in the project Git folder. Use the same facility snapshot,
seed, and training settings for all three notebooks. Match the split and holdout
fingerprints before comparing scores. See the [comparison guide](CLAUDE_HANDOFF.md#run-and-compare).

In [ ]:
%pip install "scikit-learn==1.7.2" "wandb==0.31.2" "skops==0.16.0" "safetensors==0.8.0"

In [ ]:
import os
import sys
from pathlib import Path

SEED, EPOCHS, PATIENCE = 42, 300, 30
CATALOG_SCHEMA = "databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset"
FACILITIES_TABLE = f"{CATALOG_SCHEMA}.facilities"

cwd = Path.cwd()
relative = Path("ML/models with population density")
repo_override = os.environ.get("DATATHON_REPO_ROOT")
candidates = [Path(repo_override).expanduser()] if repo_override else [cwd, *cwd.parents]
root = next((path for path in candidates if (path / relative / "density_data.py").is_file()), None)
if root is None:
    raise RuntimeError("Open the notebook in the project Git folder or set DATATHON_REPO_ROOT.")
sys.path[:0] = [str(root / relative), str(root / "ML/notebooks")]

from density_capacity import TEXT, train_capacity
from wandb_models import log_torch_capacity

## Load the facility snapshot

Rows need a usable bed label and a facility group to enter the split. Neither
density nor NFHS coverage affects who is eligible in any of the three notebooks.

In [ ]:
facilities = spark.table(FACILITIES_TABLE).toPandas()

## Fit and compare on held-out facilities

Fit the preprocessor and MLP on the training partition. Validation selects the
training checkpoint. Report MAE and RMSE in beds, plus unitless R², on the held-out partition.
The median of the training bed counts gives a simple benchmark on the same rows.

In [ ]:
result = train_capacity(
    facilities, include_nfhs=False, include_density=False,
    seed=SEED, epochs=EPOCHS, patience=PATIENCE,
)
print("Facility groups:", result["facility_counts"])
print("Split fingerprint:", result["split_fingerprint"])
print("Holdout fingerprint:", result["holdout_fingerprint"])
print("Training median baseline:", result["baseline_metrics"])
display(result["test_cohorts"])

## Save the evaluated model in Weights & Biases

The artifact contains the fitted preprocessor, network weights and aggregate
evaluation metrics and train and validation loss by epoch. The default W&B mode is offline. See the [tracking setup](../README.md).

In [ ]:
capacity_run = log_torch_capacity(
    name="capacity-mlp-facilities-only",
    model=result["model"],
    preprocessor=result["preprocessor"],
    numeric_features=result["numeric_features"],
    categorical_features=result["categorical_features"],
    metrics=result["test_metrics"],
    baseline_metrics=result["baseline_metrics"],
    history=result["training_history"],
    evaluation_tables={"test_cohorts": result["test_cohorts"].to_dict(orient="records")},
    config={
        "source_tables": [FACILITIES_TABLE],
        "text_features": list(TEXT),
        "include_density": False,
        "include_nfhs": False,
        "split_seed": SEED,
        "max_epochs": EPOCHS,
        "patience": PATIENCE,
        "epochs_run": len(result["training_history"]),
        "best_epoch": result["best_epoch"],
        "device": result["device"],
        "optimizer": "Adam",
        "loss": "Huber on log1p beds",
        "batch_size": 256,
        "learning_rate": 0.001,
        "weight_decay": 0.0001,
        "split_fingerprint": result["split_fingerprint"],
        "holdout_fingerprint": result["holdout_fingerprint"],
        "facility_counts": result["facility_counts"],
        "row_counts": result["row_counts"],
        "prediction_postprocess": "expm1, then round to nearest whole bed and clamp at zero",
    },
)
print(capacity_run)